# Beatrice Trainer for Kaggle

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/Project-Beatrice-V2/Beatrice-colab/blob/main/BeatriceV2_Trainer_Notebook_Kaggle.ipynb)

Train your own voice model on a free Kaggle GPU, then use it live in the [Beatrice voice changer](https://project-beatrice.vercel.app) for [Mac](https://github.com/satiricalguru/Beatrice-voicechanger-mac) and [Windows](https://github.com/satiricalguru/Beatrice-voicechanger-windows). This notebook runs the official [Beatrice trainer](https://huggingface.co/fierce-cats/beatrice-trainer) by Project Beatrice.

**Before you start**, in the notebook sidebar under **Session options**:

- **Accelerator:** GPU T4 x2. The trainer uses one GPU.
- **Internet:** on. This needs a Kaggle account with a verified phone number.
- **Persistence:** if your editor offers it, choose **Files**, so `/kaggle/working` survives a restart.

| Step | What it does |
|:--|:--|
| **1. Set up** | Installs the trainer. Run it after every restart. |
| **2. Add a voice** | **2a** downloads a Hugging Face dataset, **2b** copies your own recordings from a Kaggle dataset. Run one. |
| **3. TensorBoard** | Optional, and often blocked on Kaggle. |
| **4. Train** | Trains the voice. Run it again to resume after a restart. |
| **5. Export** | Builds the ZIP for **Upload Custom Model** in Beatrice. |
| **6. Clean up** | Optional. Deletes recordings and checkpoints. |

Code cells start with a few settings you can edit. Only train on voices you have the right to use.

[Website](https://project-beatrice.vercel.app) · [Notebook repo](https://github.com/Project-Beatrice-V2/Beatrice-colab) · [Ready-made voice models](https://github.com/Project-Beatrice-V2/Beatrice-voice-models) · [Report a problem](https://github.com/Project-Beatrice-V2/Beatrice-colab/issues/new)

## Step 1: Set up

In [ ]:
# 1. Set up: install the trainer
# Run this first, and again after every restart. Your voices, checkpoints and exported models live in /kaggle/working.
# trainer_revision pins the upstream trainer (huggingface.co/fierce-cats/beatrice-trainer) to the version whose
# models the Beatrice voice changer loads. Set it to "main" only if you know you need a newer one.
trainer_revision = "f34836de014b86956096878aecb8d3b17feaaa0b"

import glob, importlib.metadata, json, os, re, shutil, subprocess, sys, urllib.request, zipfile

BASE_DIR = "/kaggle/working"
TRAINER_DIR = "/tmp/beatrice-trainer"
STAGING_ROOT = "/tmp/beatrice_train_data"

try:
    urllib.request.urlopen("https://huggingface.co", timeout=15)
except Exception:
    raise RuntimeError("No internet. In the notebook sidebar turn Internet on (your Kaggle account needs a verified phone number).")

DATA_ROOT = os.path.join(BASE_DIR, "data")
OUTPUT_ROOT = os.path.join(BASE_DIR, "output")
EXPORT_ROOT = BASE_DIR
for d in (DATA_ROOT, OUTPUT_ROOT, EXPORT_ROOT):
    os.makedirs(d, exist_ok=True)


def run(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)


# ---- PyTorch: the trainer needs torchaudio >= 2.1 and < 2.9 ----------------------------------
def _version(pkg):
    try:
        return importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        return None


def _major_minor(v):
    return tuple(int(x) for x in re.findall(r"\d+", v)[:2]) if v else None


torch_v, audio_v = _version("torch"), _version("torchaudio")
audio_ok = (audio_v is not None and (2, 1) <= _major_minor(audio_v) < (2, 9)
            and torch_v is not None and torch_v.split("+")[0] == audio_v.split("+")[0])
print(f"Found torch {torch_v}, torchaudio {audio_v}")
if not audio_ok:
    print("Installing torch 2.8.0 + torchaudio 2.8.0 (CUDA 12.6), the newest pair the trainer supports. This takes a few minutes...")
    run(f'"{sys.executable}" -m pip install -q torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 '
        '--index-url https://download.pytorch.org/whl/cu126')

# ---- Other trainer dependencies (pyworld is compiled from source, give it a minute) ----------
run(f'"{sys.executable}" -m pip install -q "pyworld>=0.3.4" "soundfile>=0.11" "tensorboard>=2" "tqdm>=4" datasets huggingface_hub')

# ---- The upstream Beatrice trainer, pinned to the revision the voice changer can load ----------
def _trainer_ready():
    cfg = os.path.join(TRAINER_DIR, "assets", "default_config.json")
    if not os.path.isfile(os.path.join(TRAINER_DIR, "beatrice_trainer", "__main__.py")) or not os.path.isfile(cfg):
        return False
    with open(cfg) as f:
        c = json.load(f)
    for key in ("phone_extractor_file", "pitch_estimator_file", "pretrained_file"):
        p = os.path.join(TRAINER_DIR, c[key])
        if not os.path.isfile(p) or os.path.getsize(p) < 1_000_000:  # missing, or a Git LFS pointer stub
            return False
    return True


trainer_revision = trainer_revision.strip() or "main"
head = subprocess.run(["git", "-C", TRAINER_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
if not (_trainer_ready() and (trainer_revision == "main" or head.startswith(trainer_revision))):
    if shutil.which("git-lfs") is None:
        run("apt-get -qq update && apt-get -qq install -y git-lfs")
    run("git lfs install")
    shutil.rmtree(TRAINER_DIR, ignore_errors=True)
    run(f'git clone -q --no-checkout https://huggingface.co/fierce-cats/beatrice-trainer "{TRAINER_DIR}"')
    run(f'git -C "{TRAINER_DIR}" checkout -q {trainer_revision}')
    if not _trainer_ready():
        run(f'git -C "{TRAINER_DIR}" lfs pull')
    if not _trainer_ready():
        raise RuntimeError("The trainer's pretrained files did not download (Git LFS). Re-run this step.")

with open(os.path.join(TRAINER_DIR, "beatrice_trainer", "__main__.py"), encoding="utf-8") as f:
    m = re.search(r'PARAPHERNALIA_VERSION = "([^"]+)"', f.read())
model_format = m.group(1) if m else "unknown"
if model_format != "2.0.0-rc.0":
    print(f"Warning: this trainer writes model format {model_format}. "
          "The Beatrice voice changer loads 2.0.0-rc.0 models, so the result may not import.")

# ---- Check the environment in a fresh process (the trainer runs as a separate process too) ----
check = subprocess.run([sys.executable, "-c", (
    "import torch, torchaudio, pyworld, soundfile\n"
    "assert 'soundfile' in torchaudio.list_audio_backends(), 'torchaudio cannot use soundfile'\n"
    "gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'\n"
    "cap = '.'.join(map(str, torch.cuda.get_device_capability(0))) if torch.cuda.is_available() else '-'\n"
    "print(f'torch {torch.__version__} | torchaudio {torchaudio.__version__} | GPU {gpu} (compute {cap})')\n"
)], capture_output=True, text=True)
print(check.stdout.strip() or check.stderr.strip()[-2000:])
if check.returncode != 0:
    raise RuntimeError("The training environment is not working. See the error above.")
if "GPU none" in check.stdout:
    print("Warning: no GPU found. Training on CPU is extremely slow. In the sidebar set Accelerator to GPU T4 x2, then run this step again.")
elif re.search(r"compute [0-6]\.", check.stdout):
    print("Warning: this GPU is older than the T4. If training fails with 'no kernel image is available', set Accelerator to GPU T4 x2.")

# ---- Helpers used by the later steps --------------------------------------------------------
AUDIO_EXTS = (".wav", ".flac", ".fla", ".mp3", ".ogg", ".oga", ".opus", ".aif", ".aiff")
MODEL_FILES = ("phone_extractor.bin", "pitch_estimator.bin", "waveform_generator.bin",
               "embedding_setter.bin", "speaker_embeddings.bin")


def clean_name(name):
    """A safe folder name for a voice (letters, digits, '-', '_', '.')."""
    name = re.sub(r"[^\w.-]+", "_", str(name).strip()).strip("._")
    if not name:
        raise ValueError("Give the voice a name, for example my_voice.")
    return name


def _unique_path(folder, filename):
    stem, ext = os.path.splitext(filename)
    path, k = os.path.join(folder, filename), 1
    while os.path.exists(path):
        path, k = os.path.join(folder, f"{stem}_{k}{ext}"), k + 1
    return path


def import_audio(src, dest):
    """Copy every audio file in src (a folder, a ZIP, or one file) into dest. ZIPs are unpacked."""
    os.makedirs(dest, exist_ok=True)
    paths = [src] if os.path.isfile(src) else sorted(glob.glob(os.path.join(src, "**", "*"), recursive=True))
    count = 0
    for path in paths:
        base = os.path.basename(path)
        if not os.path.isfile(path) or base.startswith(".") or "__MACOSX" in path:
            continue
        if base.lower().endswith(".zip"):
            with zipfile.ZipFile(path) as zf:
                for member in zf.infolist():
                    name = os.path.basename(member.filename)
                    if (member.is_dir() or "__MACOSX" in member.filename or name.startswith(".")
                            or not name.lower().endswith(AUDIO_EXTS)):
                        continue
                    with zf.open(member) as s, open(_unique_path(dest, name), "wb") as d:
                        shutil.copyfileobj(s, d)
                    count += 1
        elif base.lower().endswith(AUDIO_EXTS):
            shutil.copyfile(path, _unique_path(dest, base))
            count += 1
    return count


def describe_audio(folder):
    """Print how many audio files a voice folder has and how long they are."""
    files = [f for f in glob.glob(os.path.join(folder, "**", "*"), recursive=True)
             if f.lower().endswith(AUDIO_EXTS)]
    seconds, unreadable = 0.0, 0
    try:
        import soundfile as sf
        for f in files:
            try:
                seconds += sf.info(f).duration
            except Exception:
                unreadable += 1
    except ImportError:
        unreadable = len(files)
    print(f"Voice folder: {folder}")
    print(f"Audio files:  {len(files)}  ({seconds / 60:.1f} minutes)")
    if unreadable:
        print(f"Warning: {unreadable} file(s) could not be read and may break training. Remove or convert them.")
    if not files:
        print("No audio found. Use WAV, FLAC, MP3, OGG/Opus or AIFF files.")
    return len(files)


def remember_voice(name):
    with open(os.path.join(BASE_DIR, ".last_voice"), "w") as f:
        f.write(name)


def pick_voice(name, root):
    """Resolve the voice to use: the given name, else the one from Step 2, else the only one."""
    voices = sorted(d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))) if os.path.isdir(root) else []
    if str(name).strip():
        name = clean_name(name)
    else:
        last = os.path.join(BASE_DIR, ".last_voice")
        name = open(last).read().strip() if os.path.isfile(last) else (voices[0] if len(voices) == 1 else "")
    if name not in voices:
        raise ValueError(f"Voice '{name}' not found in {root}. Available: {', '.join(voices) or 'none yet (run Step 2)'}")
    return name


def find_models(out_dir):
    """Paraphernalia folders the trainer wrote, as (step, path), oldest first."""
    found = []
    for path in glob.glob(os.path.join(out_dir, "paraphernalia_*")):
        m = re.search(r"_(\d{8})(?:_[0-9a-f]+)?$", os.path.basename(path))
        if os.path.isdir(path) and m:
            found.append((int(m.group(1)), os.path.getmtime(path), path))
    return [(step, path) for step, _, path in sorted(found)]


def build_model_zip(model_dir, voice, step, dest_dir):
    """ZIP one paraphernalia folder as beatrice_paraphernalia_<voice>_<step>/ for Upload Custom Model."""
    missing = [f for f in MODEL_FILES if not os.path.isfile(os.path.join(model_dir, f))]
    if missing:
        raise RuntimeError(f"{model_dir} is incomplete, missing: {', '.join(missing)}")
    folder = f"beatrice_paraphernalia_{voice}_{step:08d}"
    os.makedirs(dest_dir, exist_ok=True)
    zip_path = os.path.join(dest_dir, folder + ".zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for name in sorted(os.listdir(model_dir)):
            if os.path.isfile(os.path.join(model_dir, name)) and not name.startswith("."):
                zf.write(os.path.join(model_dir, name), f"{folder}/{name}")
    return zip_path


BEATRICE_SETUP_DONE = True

print()
print(f"Ready. Trainer {trainer_revision[:12]} (model format {model_format}) in {TRAINER_DIR}")
print(f"Data folder: {DATA_ROOT}")
print("Next: Step 2a or 2b to add a voice.")

## Step 2: Add a voice
Run **2a** to download a dataset from Hugging Face, or **2b** to copy your own recordings from a Kaggle dataset. Each voice gets its own folder, named by `speaker_name`.

In [ ]:
# 2a. Download a voice dataset from Hugging Face
# A public Hugging Face dataset with an audio column, or a dataset repo of audio files or ZIPs.
# Audio is saved at its original quality.
huggingface_repo_id = "meoconxinhxan/trump-speech-dataset-tts"
split = "train"
speaker_name = "trump"

if "BEATRICE_SETUP_DONE" not in globals():
    raise RuntimeError("Run Step 1 first. It is needed after every restart.")

voice = clean_name(speaker_name)
voice_dir = os.path.join(DATA_ROOT, voice)
shutil.rmtree(voice_dir, ignore_errors=True)
os.makedirs(voice_dir)
print(f"Downloading {huggingface_repo_id} ...")

saved = 0
try:
    from datasets import Audio, load_dataset
    ds = load_dataset(huggingface_repo_id, split=split)
    audio_cols = [c for c, feature in ds.features.items() if isinstance(feature, Audio)]
    if not audio_cols:
        raise ValueError("no audio column")
    col = audio_cols[0]
    ds = ds.cast_column(col, Audio(decode=False))  # keep the original files, no resampling
    for i, item in enumerate(ds):
        audio = item[col] or {}
        ext = os.path.splitext(audio.get("path") or "")[1].lower()
        target = os.path.join(voice_dir, f"{voice}_{i:05d}{ext if ext in AUDIO_EXTS else '.wav'}")
        if audio.get("bytes"):
            with open(target, "wb") as f:
                f.write(audio["bytes"])
        elif audio.get("path") and os.path.isfile(audio["path"]):
            shutil.copyfile(audio["path"], target)
        else:
            continue
        saved += 1
except Exception as e:
    print(f"Not a standard audio dataset ({e}). Downloading the repository files instead...")
    from huggingface_hub import snapshot_download
    tmp = "/tmp/hf_download"
    shutil.rmtree(tmp, ignore_errors=True)
    snapshot_download(repo_id=huggingface_repo_id, repo_type="dataset", local_dir=tmp)
    saved = import_audio(tmp, voice_dir)
    shutil.rmtree(tmp, ignore_errors=True)

print()
if describe_audio(voice_dir):
    remember_voice(voice)
    print(f"\nVoice '{voice}' is ready. Next: Step 4 (start Step 3 first if you want TensorBoard).")

In [ ]:
# 2b. Use your own recordings from a Kaggle dataset
# Upload your audio as a Kaggle dataset: sidebar > Add Input > Upload > New Dataset. Kaggle unpacks ZIPs.
# Then paste the dataset's path below. It is shown under Input in the sidebar (the copy icon copies it).
# Clean recordings of ONE speaker, no music or reverb: WAV, FLAC, MP3, OGG/Opus or AIFF.
kaggle_dataset_path = "/kaggle/input/your-dataset-name"
speaker_name = "my_voice"
add_to_existing = False

if "BEATRICE_SETUP_DONE" not in globals():
    raise RuntimeError("Run Step 1 first. It is needed after every restart.")

voice = clean_name(speaker_name)
voice_dir = os.path.join(DATA_ROOT, voice)
if not os.path.exists(kaggle_dataset_path):
    inputs = sorted(glob.glob("/kaggle/input/*") + glob.glob("/kaggle/input/*/*"))
    raise FileNotFoundError(f"{kaggle_dataset_path} not found. Inputs attached to this notebook: {inputs or 'none'}")
if not add_to_existing:
    shutil.rmtree(voice_dir, ignore_errors=True)
added = import_audio(kaggle_dataset_path, voice_dir)

print(f"Added {added} audio file(s).\n")
if describe_audio(voice_dir):
    remember_voice(voice)
    print(f"\nVoice '{voice}' is ready. Next: Step 4 (start Step 3 first if you want TensorBoard).")

## Step 3: TensorBoard (optional)

In [ ]:
# 3. Watch training in TensorBoard (optional)
# Start this before Step 4. Kaggle often blocks the embedded TensorBoard view; if it stays blank, skip this step.
# The test conversions are also saved as TensorBoard event files in /kaggle/working/output/<voice>.
if "BEATRICE_SETUP_DONE" not in globals():
    raise RuntimeError("Run Step 1 first. It is needed after every restart.")
%load_ext tensorboard
%tensorboard --logdir "{OUTPUT_ROOT}"

## Step 4: Train

In [ ]:
# 4. Train
# Trains one voice. Checkpoints are saved every save_every_n_steps, so after a restart run Step 1 and this step
# again: it resumes from the last checkpoint. To train longer later, raise training_steps and run it again.
# speaker_name: leave empty to train the voice from Step 2. num_workers: 0 picks one per CPU core.
speaker_name = ""
training_steps = 5000
batch_size = 8
learning_rate = 0.00005
save_every_n_steps = 1000
warmup_steps = 1000
num_workers = 0
resume_from_checkpoint = True
log_loss_curves = False

if "BEATRICE_SETUP_DONE" not in globals():
    raise RuntimeError("Run Step 1 first. It is needed after every restart.")
import time

voice = pick_voice(speaker_name, DATA_ROOT)
voice_dir = os.path.join(DATA_ROOT, voice)
if not describe_audio(voice_dir):
    raise RuntimeError("This voice has no audio. Run Step 2 again.")

# The trainer wants <data_dir>/<speaker>/<audio>. Stage a local copy named after the voice, so the model
# comes out as paraphernalia_<voice>_<step> (on Colab this is also faster than reading from Drive).
run_data_dir = os.path.join(STAGING_ROOT, voice)
shutil.rmtree(run_data_dir, ignore_errors=True)
shutil.copytree(voice_dir, os.path.join(run_data_dir, voice))

# One output folder per voice, so checkpoints of different voices never mix.
out_dir = os.path.join(OUTPUT_ROOT, voice)
resume_flag = ""
if os.path.isfile(os.path.join(out_dir, "checkpoint_latest.pt.gz")):
    if resume_from_checkpoint:
        resume_flag = "-r"
        print(f"Resuming from {out_dir}/checkpoint_latest.pt.gz")
    else:
        backup = f"{out_dir}_old_{time.strftime('%Y%m%d-%H%M%S')}"
        os.rename(out_dir, backup)
        print(f"Starting fresh. The previous run was moved to {backup}")
elif os.path.isfile(os.path.join(OUTPUT_ROOT, "checkpoint_latest.pt.gz")):
    print(f"Note: {OUTPUT_ROOT}/checkpoint_latest.pt.gz is from an older version of this notebook. "
          f"To resume it, move the files in {OUTPUT_ROOT} into {out_dir} and run this step again.")

with open(os.path.join(TRAINER_DIR, "assets", "default_config.json")) as f:
    config = json.load(f)
config.update({
    "n_steps": training_steps,
    "batch_size": batch_size,
    "learning_rate_g": learning_rate,
    "learning_rate_d": learning_rate,
    "save_interval": save_every_n_steps,
    "evaluation_interval": save_every_n_steps,
    "warmup_steps": warmup_steps,
    "num_workers": num_workers if num_workers > 0 else (os.cpu_count() or 2),
    "record_metrics": log_loss_curves,
})
config_path = os.path.join(STAGING_ROOT, f"config_{voice}.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"\nTraining '{voice}' for {training_steps} steps (batch {batch_size}, workers {config['num_workers']}).")
print("If it stops right away after resuming, the checkpoint already reached training_steps: raise it.\n")
!cd "{TRAINER_DIR}" && "{sys.executable}" -u beatrice_trainer -d "{run_data_dir}" -o "{out_dir}" -c "{config_path}" {resume_flag}

exit_code = globals().get("_exit_code", 0)
models = find_models(out_dir)
if exit_code:
    print(f"\nTraining stopped with exit code {exit_code}. Scroll up for the error. Saved checkpoints are kept.")
if models:
    print(f"\nLatest model: {models[-1][1]} (step {models[-1][0]}). Next: Step 5 to export it.")

## Step 5: Export for the Beatrice voice changer

In [ ]:
# 5. Export for the Beatrice voice changer
# Builds beatrice_paraphernalia_<voice>_<step>.zip in /kaggle/working: the ZIP you pick under
# "Upload Custom Model" in Beatrice. speaker_name: empty = the voice from Step 2. checkpoint_step: 0 = latest.
# download_type: "Voice changer ZIP (recommended)", "Training checkpoint (.pt.gz)" or "Whole output folder (large)"
speaker_name = ""
checkpoint_step = 0
download_type = "Voice changer ZIP (recommended)"

if "BEATRICE_SETUP_DONE" not in globals():
    raise RuntimeError("Run Step 1 first. It is needed after every restart.")
voice = pick_voice(speaker_name, OUTPUT_ROOT)
out_dir = os.path.join(OUTPUT_ROOT, voice)

if download_type.startswith("Voice changer"):
    models = find_models(out_dir)
    if checkpoint_step:
        models = [m for m in models if m[0] == checkpoint_step]
    if not models:
        raise RuntimeError(f"No saved model{' for step ' + str(checkpoint_step) if checkpoint_step else ''} in {out_dir}. "
                           f"Saved steps: {[s for s, _ in find_models(out_dir)] or 'none (run Step 4)'}")
    step, model_dir = models[-1]
    path = build_model_zip(model_dir, voice, step, EXPORT_ROOT)
    print(f"Built {path} from {os.path.basename(model_dir)}")
elif download_type.startswith("Training checkpoint"):
    path = os.path.join(out_dir, "checkpoint_latest.pt.gz")
    if not os.path.isfile(path):
        raise RuntimeError(f"No checkpoint in {out_dir}. Run Step 4 first.")
else:
    path = shutil.make_archive(os.path.join(EXPORT_ROOT, f"{voice}_output"), "zip", out_dir)

print(f"Size: {os.path.getsize(path) / 1e6:.1f} MB")
from IPython.display import FileLink, display
os.chdir(BASE_DIR)  # FileLink needs a path relative to /kaggle/working
display(FileLink(os.path.relpath(path, BASE_DIR)))
print("\nIf the link does not download, open Output (/kaggle/working) in the sidebar, click ... next to the file > Download.")
if download_type.startswith("Voice changer"):
    print("In Beatrice: model menu > Upload Custom Model > pick the ZIP > name it, choose a category > Import.")

## Import the model into Beatrice

1. Open the Beatrice voice changer ([Mac](https://github.com/satiricalguru/Beatrice-voicechanger-mac) or [Windows](https://github.com/satiricalguru/Beatrice-voicechanger-windows)).
2. Open the model menu and choose **Upload Custom Model**.
3. Pick the `beatrice_paraphernalia_<voice>_<step>.zip` from Step 5 (download it from the link or the Output panel), give it a name, choose a category and click **Import**.
4. Press the power button so it says **LIVE**, and talk.

Use the voice changer ZIP from Step 5, not the checkpoint or the whole output folder: Beatrice only imports the model files.

## Step 6: Clean up (optional)

In [ ]:
# 6. Delete data to free space (optional)
# Deletes files from /kaggle/working. delete_what: "One voice" (its recordings, checkpoints and
# exported ZIPs) or "Everything". Set confirm_deletion = True to run.
delete_what = "One voice"
speaker_name = ""
confirm_deletion = False

if "BEATRICE_SETUP_DONE" not in globals():
    raise RuntimeError("Run Step 1 first. It is needed after every restart.")


def remove(path):
    if os.path.isdir(path):
        shutil.rmtree(path)
        print("Deleted", path)
    elif os.path.isfile(path):
        os.remove(path)
        print("Deleted", path)


if not confirm_deletion:
    print("Nothing deleted. Set confirm_deletion = True to delete.")
elif delete_what == "Everything":
    for path in (DATA_ROOT, OUTPUT_ROOT, STAGING_ROOT, os.path.join(BASE_DIR, ".last_voice")):
        remove(path)
    for path in glob.glob(os.path.join(EXPORT_ROOT, "beatrice_paraphernalia_*.zip")) + \
            glob.glob(os.path.join(EXPORT_ROOT, "*_output.zip")):
        remove(path)
    os.makedirs(DATA_ROOT, exist_ok=True)
    os.makedirs(OUTPUT_ROOT, exist_ok=True)
    print("Done.")
else:
    voice = clean_name(speaker_name)
    for path in (os.path.join(DATA_ROOT, voice), os.path.join(OUTPUT_ROOT, voice), os.path.join(STAGING_ROOT, voice)):
        remove(path)
    for path in glob.glob(os.path.join(EXPORT_ROOT, f"beatrice_paraphernalia_{voice}_*.zip")) + \
            glob.glob(os.path.join(EXPORT_ROOT, f"{voice}_output.zip")):
        remove(path)
    print("Done.")